In [70]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("Libraries Imported Successfully")

In [71]:
session = sagemaker.Session()

bucket = session.default_bucket()
region = boto3.Session().region_name

print("Region :", region)
print("Default Bucket :", bucket)

In [72]:
import sagemaker

print("Version:", getattr(sagemaker, "__version__", "No version"))
print("Location:", sagemaker.__file__)
print(dir(sagemaker))

In [73]:
bucket = "workforce-analytics-data-infosys-virtual"
key = "master_cleaned_final.csv"

s3 = boto3.client("s3")

obj = s3.get_object(
    Bucket=bucket,
    Key=key
)

df = pd.read_csv(obj["Body"])

print(df.shape)

df.head()

In [74]:
date_cols = [
    "StartDate",
    "ExitDate",
    "DateOfBirth",
    "SurveyDate",
    "Training_Date",
    "Recruitment_ApplicationDate",
    "Recruitment_DateofBirth"
]

for col in date_cols:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        df[col] = df[col].map(
            lambda x: x.toordinal()
            if pd.notnull(x)
            else 0
        )

In [75]:
drop_cols = [
    "EmployeeID",
    "FirstName",
    "LastName",
    "ADEmail",
    "Supervisor",
    "Recruitment_FirstName",
    "Recruitment_LastName",
    "Recruitment_Email",
    "Recruitment_PhoneNumber",
    "Recruitment_Address",
    "Recruitment_ZipCode"
]

df.drop(
    columns=drop_cols,
    inplace=True,
    errors="ignore"
)

print(df.shape)

In [76]:
for col in df.columns:

    if df[col].dtype == "object":

        df[col] = df[col].fillna("Unknown")

    else:

        df[col] = df[col].fillna(
            df[col].median()
        )

In [77]:
encoders = {}

for col in df.columns:

    if df[col].dtype == "object":

        le = LabelEncoder()

        df[col] = le.fit_transform(
            df[col].astype(str)
        )

        encoders[col] = le

In [78]:
target = "EmployeeStatus"

X = df.drop(columns=[target])

y = df[target]

print(y.value_counts())

In [79]:
# ===============================
# Cell 9: Train-Test Split
# ===============================

from sklearn.model_selection import train_test_split

# Select target column
target = "EmployeeStatus"

# Features
X = df.drop(columns=[target])

# Target
y = df[target]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Data Shape :", X_train.shape)
print("Testing Data Shape  :", X_test.shape)
print("\nTraining Target Distribution")
print(y_train.value_counts())

print("\nTesting Target Distribution")
print(y_test.value_counts())

In [80]:
# ===============================
# Cell 10: Logistic Regression
# ===============================

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Scale data for Logistic Regression
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression
lr_model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_pred = lr_model.predict(X_test_scaled)

# Accuracy
lr_accuracy = accuracy_score(y_test, lr_pred)

print("========== Logistic Regression ==========")
print("Accuracy :", round(lr_accuracy * 100, 2), "%")

print("\nClassification Report")
print(classification_report(y_test, lr_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, lr_pred))

In [81]:
# ===============================
# Cell 12: XGBoost Classifier
# ===============================

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# Number of classes
num_classes = len(np.unique(y_train))

# Create XGBoost model
xgb_model = XGBClassifier(
    objective="multi:softmax",
    num_class=num_classes,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# Train the model
xgb_model.fit(X_train, y_train)

# Predictions
xgb_pred = xgb_model.predict(X_test)

# Accuracy
xgb_accuracy = accuracy_score(y_test, xgb_pred)

print("========== XGBoost ==========")
print("Accuracy :", round(xgb_accuracy * 100, 2), "%")

print("\nClassification Report")
print(classification_report(y_test, xgb_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, xgb_pred))

In [82]:
# ===============================
# Cell 11: Random Forest
# ===============================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

# Train the model
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)

# Accuracy
rf_accuracy = accuracy_score(y_test, rf_pred)

print("========== Random Forest ==========")
print("Accuracy :", round(rf_accuracy * 100, 2), "%")

print("\nClassification Report")
print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, rf_pred))

In [83]:
# ===============================
# Cell 12: XGBoost Classifier
# ===============================

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# Create XGBoost model
xgb_model = XGBClassifier(
    objective="multi:softmax",
    num_class=len(np.unique(y_train)),
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

# Train the model
xgb_model.fit(X_train, y_train)

# Predict
xgb_pred = xgb_model.predict(X_test)

# Accuracy
xgb_accuracy = accuracy_score(y_test, xgb_pred)

print("=" * 50)
print("XGBoost Results")
print("=" * 50)

print(f"Accuracy : {xgb_accuracy * 100:.2f}%")

print("\nClassification Report")
print(classification_report(y_test, xgb_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, xgb_pred))

In [84]:
# ===============================
# Cell 13: Compare All Models
# ===============================

import pandas as pd

# Store accuracies
accuracy_results = {
    "Logistic Regression": lr_accuracy,
    "Random Forest": rf_accuracy,
    "XGBoost": xgb_accuracy
}

# Create DataFrame
results = pd.DataFrame(
    list(accuracy_results.items()),
    columns=["Model", "Accuracy"]
)

# Convert to percentage
results["Accuracy (%)"] = results["Accuracy"] * 100

# Sort by highest accuracy
results = results.sort_values(
    by="Accuracy (%)",
    ascending=False
).reset_index(drop=True)

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

print(results)

print("\nBest Model :", results.iloc[0]["Model"])
print("Best Accuracy : {:.2f}%".format(results.iloc[0]["Accuracy (%)"]))

# Select the best model
if results.iloc[0]["Model"] == "Logistic Regression":
    best_model = lr_model

elif results.iloc[0]["Model"] == "Random Forest":
    best_model = rf_model

else:
    best_model = xgb_model

In [85]:
# ===============================
# Cell 14: Save the Best Model
# ===============================

import os
import joblib

# Create a folder to store models
model_dir = "saved_model"

os.makedirs(model_dir, exist_ok=True)

# Save the best model
joblib.dump(
    best_model,
    os.path.join(model_dir, "best_model.pkl")
)

# Save the scaler
joblib.dump(
    scaler,
    os.path.join(model_dir, "scaler.pkl")
)

# Save the label encoders
joblib.dump(
    encoders,
    os.path.join(model_dir, "label_encoders.pkl")
)

print("=" * 50)
print("Model Saved Successfully")
print("=" * 50)

print("Best Model :", results.iloc[0]["Model"])
print("Location   :", model_dir)

print("\nSaved Files:")

for file in os.listdir(model_dir):
    print(file)

In [86]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("saved_model", arcname=".")

print("model.tar.gz created successfully!")

In [87]:
import boto3

bucket_name = "workforce-analytics-data-infosys-virtual"
s3_key = "models/model.tar.gz"

s3 = boto3.client("s3")

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    s3_key
)

print(f"Uploaded model.tar.gz to s3://{bucket_name}/{s3_key}")

In [88]:
import os

print(os.path.exists("inference.py"))

In [89]:
import sagemaker

role = sagemaker.get_execution_role()
print(role)

In [90]:
import boto3

sm = boto3.client("sagemaker")

try:
    sm.delete_endpoint(EndpointName="workforce-analytics-endpoint")
    print("Endpoint deleted.")
except Exception as e:
    print(e)

In [91]:
import boto3

sm = boto3.client("sagemaker")

try:
    sm.delete_endpoint_config(
        EndpointConfigName="workforce-analytics-endpoint"
    )
    print("Endpoint configuration deleted.")
except Exception as e:
    print(e)

In [102]:
with open("inference.py", "r") as f:
    print(f.read())

In [103]:
import tarfile

with tarfile.open("model.tar.gz", "r:gz") as tar:
    print("Contents of model.tar.gz:")
    print(tar.getnames())

In [107]:
import joblib

model = joblib.load("saved_model/best_model.pkl")

print("Number of input features expected:", model.n_features_in_)

In [108]:
print(X.columns.tolist())

In [109]:
print("Number of features:", len(X_train.columns))
print(X_train.columns.tolist())

In [110]:
import joblib

label_encoders = joblib.load("saved_model/label_encoders.pkl")

print(type(label_encoders))
print(label_encoders.keys())

In [111]:
import joblib

scaler = joblib.load("saved_model/scaler.pkl")

print(type(scaler))

In [112]:
import os
import json
import joblib
import pandas as pd

MODEL = None
SCALER = None
ENCODERS = None

FEATURES = [
    'EmployeeID', 'FirstName', 'LastName', 'StartDate', 'ExitDate',
    'Title', 'Supervisor', 'ADEmail', 'BusinessUnit', 'EmployeeType',
    'PayZone', 'EmployeeClassificationType', 'TerminationType',
    'TerminationDescription', 'DepartmentType', 'Division',
    'DateOfBirth', 'EmployeeState', 'JobFunctionDescription',
    'Gender', 'LocationCode', 'Race', 'MaritalStatus',
    'PerformanceScore', 'CurrentEmployeeRating', 'SurveyDate',
    'EngagementScore', 'SatisfactionScore', 'WorkLifeBalanceScore',
    'Training_Date', 'Training_ProgramName', 'Training_Type',
    'Training_Outcome', 'Training_Location', 'Training_Trainer',
    'Training_DurationDays', 'Training_Cost',
    'Recruitment_ApplicationDate', 'Recruitment_FirstName',
    'Recruitment_LastName', 'Recruitment_Gender',
    'Recruitment_DateofBirth', 'Recruitment_PhoneNumber',
    'Recruitment_Email', 'Recruitment_Address',
    'Recruitment_City', 'Recruitment_State',
    'Recruitment_ZipCode', 'Recruitment_Country',
    'Recruitment_EducationLevel',
    'Recruitment_YearsofExperience',
    'Recruitment_DesiredSalary',
    'Recruitment_JobTitle',
    'Recruitment_Status'
]


def model_fn(model_dir):
    global MODEL, SCALER, ENCODERS

    MODEL = joblib.load(os.path.join(model_dir, "best_model.pkl"))
    SCALER = joblib.load(os.path.join(model_dir, "scaler.pkl"))
    ENCODERS = joblib.load(os.path.join(model_dir, "label_encoders.pkl"))

    return MODEL


def input_fn(request_body, request_content_type):
    if request_content_type != "application/json":
        raise ValueError("Only application/json is supported")

    data = json.loads(request_body)

    df = pd.DataFrame([data])

    df = df[FEATURES]

    for col, encoder in ENCODERS.items():
        if col in df.columns:
            df[col] = encoder.transform(df[col].astype(str))

    return df


def predict_fn(input_data, model):
    scaled = SCALER.transform(input_data)
    prediction = model.predict(scaled)
    return prediction


def output_fn(prediction, accept):
    return json.dumps(prediction.tolist()), "application/json"

In [113]:
df.dtypes

In [114]:
import os

print(os.listdir("saved_model"))

In [115]:
import shutil

shutil.copy("inference.py", "saved_model/inference.py")

print("Copied successfully!")

In [116]:
import os

print(os.listdir("saved_model"))

In [117]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("saved_model/best_model.pkl", arcname="best_model.pkl")
    tar.add("saved_model/scaler.pkl", arcname="scaler.pkl")
    tar.add("saved_model/label_encoders.pkl", arcname="label_encoders.pkl")
    tar.add("saved_model/inference.py", arcname="inference.py")

print("model.tar.gz created successfully")

In [119]:
import sagemaker

sess = sagemaker.Session()

model_artifact = sess.upload_data(
    path="model.tar.gz",
    bucket="workforce-analytics-data-infosys-virtual",
    key_prefix="models"
)

print(model_artifact)

In [120]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data="s3://workforce-analytics-data-infosys-virtual/models/model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sess
)

In [121]:
print(model)

In [122]:
print(role)

In [123]:
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="workforce-analytics-endpoint",
    update_endpoint=True
)

In [132]:
from sagemaker.serializers import JSONSerializer

predictor.serializer = JSONSerializer()

In [133]:
print(predictor.serializer)

In [134]:
predictor.content_type

In [135]:
print(model.model_data)

In [136]:
import boto3

s3 = boto3.client("s3")

s3.upload_file(
    "model.tar.gz",
    "workforce-analytics-data-infosys-virtual",
    "models/model.tar.gz"
)

print("Upload completed!")

In [137]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data="s3://workforce-analytics-data-infosys-virtual/models/model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="1.2-1",
    py_version="py3",
)

In [ ]:
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="workforce-analytics-endpoint",
    update_endpoint=True
)

In [147]:
!tar -czvf model.tar.gz -C saved_model .

In [9]:
import tarfile

with tarfile.open("model.tar.gz", "r:gz") as tar:
    for name in tar.getnames():
        print(name)

In [10]:
!head -40 saved_model/code/inference.py

In [11]:
!grep -n "def model_fn" saved_model/code/inference.py

In [12]:
!grep -n "def model_fn" inference.py

In [13]:
!ls -l

In [14]:
import tarfile
import os

if os.path.exists("model.tar.gz"):
    os.remove("model.tar.gz")

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model", arcname=".")

print("model.tar.gz created successfully!")

In [20]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

In [22]:
print(type(predictor.serializer))
print(type(predictor.deserializer))

In [23]:
sample = {
    "StartDate": "",
    "ExitDate": "",
    "Title": "",
    "BusinessUnit": "",
    "EmployeeType": "",
    "PayZone": "",
    "EmployeeClassificationType": "",
    "TerminationType": "",
    "TerminationDescription": "",
    "DepartmentType": "",
    "Division": "",
    "DateOfBirth": "",
    "EmployeeState": "",
    "JobFunctionDescription": "",
    "Gender": "",
    "LocationCode": "",
    "Race": "",
    "MaritalStatus": "",
    "PerformanceScore": "",
    "CurrentEmployeeRating": 0,
    "SurveyDate": "",
    "EngagementScore": 0,
    "SatisfactionScore": 0,
    "WorkLifeBalanceScore": 0,
    "Training_Date": "",
    "Training_ProgramName": "",
    "Training_Type": "",
    "Training_Outcome": "",
    "Training_Location": "",
    "Training_Trainer": "",
    "Training_DurationDays": 0,
    "Training_Cost": 0,
    "Recruitment_ApplicationDate": "",
    "Recruitment_Gender": "",
    "Recruitment_DateofBirth": "",
    "Recruitment_City": "",
    "Recruitment_State": "",
    "Recruitment_Country": "",
    "Recruitment_EducationLevel": "",
    "Recruitment_YearsofExperience": 0,
    "Recruitment_DesiredSalary": 0,
    "Recruitment_JobTitle": "",
    "Recruitment_Status": ""
}

print(sample)

In [25]:
import joblib

model = joblib.load("saved_model/best_model.pkl")
scaler = joblib.load("saved_model/scaler.pkl")
label_encoders = joblib.load("saved_model/label_encoders.pkl")

print(type(model))
print(type(scaler))
print(type(label_encoders))

In [26]:
import pandas as pd

df = pd.DataFrame([sample])

for col, encoder in label_encoders.items():
    if col in df.columns:
        df[col] = df[col].astype(str)
        df[col] = df[col].apply(
            lambda x: x if x in encoder.classes_ else encoder.classes_[0]
        )
        df[col] = encoder.transform(df[col])

print(df.head())

In [28]:
print(df.dtypes)

In [29]:
print(scaler.feature_names_in_)

In [30]:
from sklearn.preprocessing import StandardScaler

In [31]:
scaler = StandardScaler()

In [36]:
%history -n -f history.py

In [37]:
!grep -n "train_test_split" history.py

In [38]:
df.dtypes

In [39]:
df.head()

In [40]:
from sagemaker.sklearn.model import SKLearnModel

sklearn_model = SKLearnModel(
    model_data="s3://workforce-analytics-data-infosys-virtual/models/model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="1.2-1",
    py_version="py3",
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="workforce-analytics-endpoint-v6"
)

In [44]:
print(sample)

In [46]:
%whos DataFrame

In [47]:
print(df.head())
print(df.shape)

In [48]:
import boto3

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket="workforce-analytics-data-infosys-virtual"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

In [49]:
import pandas as pd

df_original = pd.read_csv(
    "s3://workforce-analytics-data-infosys-virtual/master_cleaned_final.csv"
)

print(df_original.shape)
df_original.head()

In [50]:
sample = df_original.iloc[0].to_dict()

print(sample)

In [51]:
print(df.columns.tolist())

In [52]:
sample = df_original[df.columns].iloc[0].to_dict()

print(sample)

In [55]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("saved_model/best_model.pkl", arcname="best_model.pkl")
    tar.add("saved_model/scaler.pkl", arcname="scaler.pkl")
    tar.add("saved_model/label_encoders.pkl", arcname="label_encoders.pkl")
    tar.add("saved_model/code", arcname="code")

print("model.tar.gz created successfully")

In [56]:
import boto3

s3 = boto3.client("s3")

s3.upload_file(
    Filename="model.tar.gz",
    Bucket="workforce-analytics-data-infosys-virtual",
    Key="models/model.tar.gz"
)

print("✅ model.tar.gz uploaded successfully!")

In [57]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data="s3://workforce-analytics-data-infosys-virtual/models/model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sagemaker_session,
)

predictor = model.deploy(
    endpoint_name="workforce-analytics-endpoint-v6",
    initial_instance_count=1,
    instance_type="ml.m5.large",
    update_endpoint=True
)

print("✅ Endpoint updated successfully!")

In [58]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

print("✅ Predictor configured")

In [60]:
import joblib

scaler = joblib.load("saved_model/scaler.pkl")

print(type(scaler))

if hasattr(scaler, "feature_names_in_"):
    print(scaler.feature_names_in_)
else:
    print("No feature_names_in_ attribute")

In [61]:
import joblib
import pandas as pd

model = joblib.load("saved_model/best_model.pkl")
scaler = joblib.load("saved_model/scaler.pkl")
label_encoders = joblib.load("saved_model/label_encoders.pkl")

df = pd.DataFrame([sample])

date_cols = [
    "StartDate",
    "ExitDate",
    "DateOfBirth",
    "SurveyDate",
    "Training_Date",
    "Recruitment_ApplicationDate",
    "Recruitment_DateofBirth"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
    df[col] = df[col].map(lambda x: x.toordinal() if pd.notnull(x) else 0)

for col, encoder in label_encoders.items():
    if col in df.columns:
        df[col] = df[col].fillna("")
        df[col] = df[col].astype(str)
        df[col] = df[col].apply(
            lambda x: x if x in encoder.classes_ else encoder.classes_[0]
        )
        df[col] = encoder.transform(df[col])

print("Preprocessing successful")

X = scaler.transform(df)
print("Scaling successful")

prediction = model.predict(X)
print("Prediction:", prediction)

In [63]:
import boto3

sm = boto3.client("sagemaker")

endpoint = sm.describe_endpoint(
    EndpointName="workforce-analytics-endpoint-v6"
)

print(endpoint["EndpointConfigName"])

In [64]:
endpoint_config = sm.describe_endpoint_config(
    EndpointConfigName="sagemaker-scikit-learn-2026-07-29-16-07-44-804"
)

print(endpoint_config["ProductionVariants"][0]["ModelName"])

In [65]:
endpoint_config = sm.describe_endpoint_config(
    EndpointConfigName="sagemaker-scikit-learn-2026-07-29-16-07-44-804"
)

print(endpoint_config)

In [66]:
model_desc = sm.describe_model(
    ModelName="sagemaker-scikit-learn-2026-07-29-16-07-44-804"
)

print(model_desc["PrimaryContainer"]["ModelDataUrl"])

In [67]:
import tarfile

with tarfile.open("model.tar.gz", "r:gz") as tar:
    for member in tar.getnames():
        print(member)

In [68]:
print(predictor.endpoint_name)

In [72]:
!tar -czvf model.tar.gz best_model.pkl scaler.pkl label_encoders.pkl code

In [73]:
import os

print("Current folder:", os.getcwd())
print(os.listdir())

In [81]:
import os
import time

path = "/home/sagemaker-user/saved_model/model.tar.gz"

print(os.path.getsize(path))
print(time.ctime(os.path.getmtime(path)))

In [82]:
import boto3

s3 = boto3.client("s3")

response = s3.head_object(
    Bucket="workforce-analytics-data-infosys-virtual",
    Key="models/model.tar.gz"
)

print("LastModified:", response["LastModified"])
print("ContentLength:", response["ContentLength"])

In [83]:
import boto3

sm = boto3.client("sagemaker")

endpoint = sm.describe_endpoint(
    EndpointName="workforce-analytics-endpoint-v6"
)

print("EndpointConfig:", endpoint["EndpointConfigName"])

config = sm.describe_endpoint_config(
    EndpointConfigName=endpoint["EndpointConfigName"]
)

print(config["ProductionVariants"][0]["ModelName"])

model_name = config["ProductionVariants"][0]["ModelName"]

model = sm.describe_model(ModelName=model_name)

print(model["PrimaryContainer"]["ModelDataUrl"])
print(model["PrimaryContainer"]["Environment"])

In [84]:
import boto3
import tarfile

s3 = boto3.client("s3")

bucket = "sagemaker-eu-north-1-121470661483"
key = "sagemaker-scikit-learn-2026-07-29-16-07-44-645/sourcedir.tar.gz"

s3.download_file(bucket, key, "sourcedir.tar.gz")

with tarfile.open("sourcedir.tar.gz", "r:gz") as tar:
    print(tar.getnames())

In [85]:
import tarfile

with tarfile.open("sourcedir.tar.gz", "r:gz") as tar:
    tar.extract("inference.py")

with open("inference.py", "r") as f:
    print(f.read())

In [86]:
import boto3
import tarfile

s3 = boto3.client("s3")

bucket = "sagemaker-eu-north-1-121470661483"
key = "sagemaker-scikit-learn-2026-07-29-16-07-44-645/sourcedir.tar.gz"

s3.download_file(bucket, key, "sourcedir.tar.gz")

with tarfile.open("sourcedir.tar.gz", "r:gz") as tar:
    tar.extract("inference.py")

with open("inference.py", "r") as f:
    print(f.read())

In [87]:
!tar -czvf model.tar.gz best_model.pkl scaler.pkl label_encoders.pkl code

In [88]:
import boto3

s3 = boto3.client("s3")

s3.upload_file(
    "model.tar.gz",
    "workforce-analytics-data-infosys-virtual",
    "models/model.tar.gz"
)

print("Upload successful")

In [89]:
response = s3.head_object(
    Bucket="workforce-analytics-data-infosys-virtual",
    Key="models/model.tar.gz"
)

print("LastModified:", response["LastModified"])
print("ContentLength:", response["ContentLength"])

In [90]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
import sagemaker

sess = sagemaker.Session()
role = sagemaker.get_execution_role()

sklearn_model = SKLearnModel(
    model_data="s3://workforce-analytics-data-infosys-virtual/models/model.tar.gz",
    role=role,
    framework_version="1.2-1",
    py_version="py3",
    entry_point="inference.py",
    source_dir="code",
    sagemaker_session=sess
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="workforce-analytics-endpoint-v6",
    update_endpoint=True
)

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

print("Endpoint updated successfully!")

In [91]:
import boto3
import tarfile

sm = boto3.client("sagemaker")

endpoint = sm.describe_endpoint(
    EndpointName="workforce-analytics-endpoint-v6"
)

config = sm.describe_endpoint_config(
    EndpointConfigName=endpoint["EndpointConfigName"]
)

model_name = config["ProductionVariants"][0]["ModelName"]

model = sm.describe_model(ModelName=model_name)

print(model["PrimaryContainer"]["Environment"]["SAGEMAKER_SUBMIT_DIRECTORY"])

In [92]:
import boto3
import tarfile

s3 = boto3.client("s3")

bucket = "sagemaker-eu-north-1-121470661483"
key = "sagemaker-scikit-learn-2026-07-29-17-12-03-390/sourcedir.tar.gz"

s3.download_file(bucket, key, "sourcedir.tar.gz")

with tarfile.open("sourcedir.tar.gz", "r:gz") as tar:
    print(tar.getnames())
    tar.extract("inference.py")

with open("inference.py", "r") as f:
    print(f.read())

In [93]:
response = predictor.predict(sample)
print(response)